In [1]:
# ENVIRONMENT SETUP

!pip install biopython
!apt-get update -qq
!apt-get install -y minimap2

import os
import math
import random
import gzip
import subprocess
import time
from Bio import SeqIO
from google.colab import drive
from collections import Counter, defaultdict

drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.4 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  minimap2
0 upgraded, 1 newly installed, 0 to remove and 7 not upgraded.
Need to get 381 kB of archives.
After this operation, 497 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 minimap2 amd64 2.24+dfsg-2 [381 kB]
Fetched 381 kB in 0s (7,149 kB/s)
Selecting previously unselected package minimap2.
(Reading database ... 118242 files and directories currently installed.)
Preparing to unpack .../minimap2_2.24+dfsg-2_amd64.deb ...
Unpacking minimap2 (2.24+dfsg-2) ...
Setting up minimap2 (2.24+dfsg-2) ...
Processing triggers for man-db

In [2]:
# METAGENOME CREATION

PATH_READS = "/content/drive/Shareddrives/BIOINFORMATICS/reads"

READS_PER_BACTERIA = 5000
OUTPUT_NAME = "metagenome_final.fasta"


def generate_metagenome():
    all_reads = []

    try:
        source_files = [f for f in os.listdir(PATH_READS) if f.endswith(".fastq.gz")]
    except FileNotFoundError:
        print(f"ERROR: Cannot find the path: {PATH_READS}")
        return

    if not source_files:
        print("No .fastq.gz files found in the directory.")
        return

    for file_name in source_files:
        tag = file_name.split(".")[0]
        full_path = os.path.join(PATH_READS, file_name)

        print(f"Processing {tag}...")

        with gzip.open(full_path, "rt") as handle:
            count = 0

            for record in SeqIO.parse(handle, "fastq"):
                if count >= READS_PER_BACTERIA:
                    break

                record.id = f"{tag}|{record.id}"
                record.description = ""

                all_reads.append(record)
                count += 1

            print(f"{count} reads extracted.")

    print(f"Shuffling {len(all_reads)} reads randomly...")
    random.shuffle(all_reads)

    SeqIO.write(all_reads, OUTPUT_NAME, "fasta")

    print(f"'{OUTPUT_NAME}' has been generated.")


generate_metagenome()


# METAGENOME SAMPLE CHECK

total_count = 0

for record in SeqIO.parse(OUTPUT_NAME, "fasta"):
    if total_count < 3:
        print(f"Read {total_count + 1} Label: {record.id}")
        print(f"Sequence first 60bp: {record.seq[:60]}...")
        print("-" * 20)

    total_count += 1

print(f"\nTotal reads processed: {total_count}")

Processing blautia_coccoides...
5000 reads extracted.
Processing enterobacter_aerogenes...
5000 reads extracted.
Processing salmonella_enterica...
5000 reads extracted.
Processing escherichia_coli...
5000 reads extracted.
Processing streptococcus_infantis...
5000 reads extracted.
Shuffling 25000 reads randomly...
'metagenome_final.fasta' has been generated.
Read 1 Label: streptococcus_infantis|m160410_052906_00127_c100967582550000001823217106101632_s1_p0/5348/0_1869
Sequence first 60bp: GGGGAAGGGGAAGCCGCTTTTTCGCTTCCTGCGATTTGGTTTGATACCTCTAGGGCTTCT...
--------------------
Read 2 Label: salmonella_enterica|m140823_061001_00127_c100658602550000001823129311271454_s1_p0/7182/1997_8473
Sequence first 60bp: GACGCTGCAAAAGAGTAGAGGCGGCAGTCGTCGCTGAGAAATGGACCCTTACAACAGGCG...
--------------------
Read 3 Label: streptococcus_infantis|m160410_052906_00127_c100967582550000001823217106101632_s1_p0/4293/10733_16054
Sequence first 60bp: CCAGCTAGAAAGTAAATCATTCGGGCAATGTATTTACGGTGTTGATACTTTTCTGGTCAG...
-----

In [3]:
# K-MER CLASSIFICATION

PATH_REFS = "/content/drive/Shareddrives/BIOINFORMATICS/references/"
METAGENOME_FILE = "metagenome_final.fasta"

K = 13
VALID_BASES = {"A", "C", "G", "T"}


def get_kmer_distribution(sequence, k):
    sequence = str(sequence).upper()

    if len(sequence) < k:
        return {}

    counts = Counter()

    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i+k]

        if set(kmer).issubset(VALID_BASES):
            counts[kmer] += 1

    total = sum(counts.values())

    if total == 0:
        return {}

    return {kmer: count / total for kmer, count in counts.items()}


def vector_norm(distribution):
    return math.sqrt(sum(value * value for value in distribution.values()))


def cosine_similarity(read_dist, read_norm, ref_dist, ref_norm):
    if not read_dist or read_norm == 0.0 or ref_norm == 0.0:
        return 0.0

    dot_product = sum(value * ref_dist.get(kmer, 0.0)
                      for kmer, value in read_dist.items())

    return dot_product / (read_norm * ref_norm)

kmer_start_time = time.time()

print(f"Indexing reference genomes with K={K}...")

ref_profiles = {}
ref_norms = {}

ref_files = [f for f in os.listdir(PATH_REFS)
             if f.endswith(".fasta") or f.endswith(".fna")]

for file in ref_files:
    ref_name = file.split(".")[0]
    ref_path = os.path.join(PATH_REFS, file)

    full_counts = Counter()

    for record in SeqIO.parse(ref_path, "fasta"):
        seq = str(record.seq).upper()

        if len(seq) < K:
            continue

        for i in range(len(seq) - K + 1):
            kmer = seq[i:i+K]

            if set(kmer).issubset(VALID_BASES):
                full_counts[kmer] += 1

    total = sum(full_counts.values())

    if total == 0:
        ref_profiles[ref_name] = {}
        ref_norms[ref_name] = 0.0
        continue

    ref_dist = {kmer: count / total for kmer, count in full_counts.items()}

    ref_profiles[ref_name] = ref_dist
    ref_norms[ref_name] = vector_norm(ref_dist)

    print(f"Indexed {ref_name} ({len(ref_dist)} unique k-mers)")


print(f"\nClassifying reads from {METAGENOME_FILE}...")

kmer_results = []
correct_assignments = 0
total_reads = 0

kmer_predicted_counts = defaultdict(int)

for read in SeqIO.parse(METAGENOME_FILE, "fasta"):
    total_reads += 1

    read_dist_fwd = get_kmer_distribution(read.seq, K)
    read_dist_rev = get_kmer_distribution(read.seq.reverse_complement(), K)

    read_norm_fwd = vector_norm(read_dist_fwd)
    read_norm_rev = vector_norm(read_dist_rev)

    best_match = None
    best_score = -1.0

    for ref_name, ref_dist in ref_profiles.items():
        score_fwd = cosine_similarity(read_dist_fwd, read_norm_fwd,
                                      ref_dist, ref_norms[ref_name])

        score_rev = cosine_similarity(read_dist_rev, read_norm_rev,
                                      ref_dist, ref_norms[ref_name])

        score = max(score_fwd, score_rev)

        if score > best_score:
            best_score = score
            best_match = ref_name

    true_label = read.id.split("|")[0]

    if best_match == true_label:
        correct_assignments += 1

    kmer_predicted_counts[best_match] += 1

    kmer_results.append({
        "read_id": read.id,
        "predicted": best_match,
        "actual": true_label,
        "score": best_score
    })


accuracy = (correct_assignments / total_reads) * 100 if total_reads > 0 else 0.0

print("\nK-MER CLASSIFICATION REPORT")
print(f"Total reads: {total_reads}")
print(f"Correctly classified: {correct_assignments}")
print(f"Accuracy: {accuracy:.2f}%")

print("\nK-MER METAGENOMIC SAMPLE COMPOSITION")
for genome, count in sorted(kmer_predicted_counts.items()):
    print(f"{genome}: {count} reads")

kmer_end_time = time.time()

kmer_runtime = kmer_end_time - kmer_start_time

print(f"\nK-MER Runtime: {kmer_runtime:.2f} seconds")

Indexing reference genomes with K=13...
Indexed enterobacter_aerogenes (4558491 unique k-mers)
Indexed escherichia_coli (4170362 unique k-mers)
Indexed streptococcus_infantis (1781898 unique k-mers)
Indexed salmonella_enterica (4591133 unique k-mers)
Indexed blautia_coccoides (4582941 unique k-mers)

Classifying reads from metagenome_final.fasta...

K-MER CLASSIFICATION REPORT
Total reads: 25000
Correctly classified: 20217
Accuracy: 80.87%

K-MER METAGENOMIC SAMPLE COMPOSITION
blautia_coccoides: 7323 reads
enterobacter_aerogenes: 2718 reads
escherichia_coli: 4832 reads
salmonella_enterica: 4956 reads
streptococcus_infantis: 5171 reads

K-MER Runtime: 819.82 seconds


In [4]:
# MINIMAP2 ALIGNMENT-BASED CLASSIFICATION

PATH_REFS = "/content/drive/Shareddrives/BIOINFORMATICS/references/"
METAGENOME_FILE = "metagenome_final.fasta"

MINIMAP_OUTPUT_DIR = "minimap2_outputs"
os.makedirs(MINIMAP_OUTPUT_DIR, exist_ok=True)

ref_files = [f for f in os.listdir(PATH_REFS)
             if f.endswith(".fasta") or f.endswith(".fna")]

minimap_best_results = {}
minimap_start_time = time.time()

print("Running minimap2 for each reference genome...")

for ref_file in ref_files:
    ref_name = ref_file.split(".")[0]
    ref_path = os.path.join(PATH_REFS, ref_file)

    output_paf = os.path.join(MINIMAP_OUTPUT_DIR, f"{ref_name}.paf")

    command = f"minimap2 -x map-pb {ref_path} {METAGENOME_FILE} > {output_paf}"

    print(f"Running minimap2 for {ref_name}...")
    subprocess.run(command, shell=True)

    with open(output_paf, "r") as paf_file:
        for line in paf_file:
            fields = line.strip().split("\t")

            if len(fields) < 12:
                continue

            read_id = fields[0]
            target_name = ref_name

            matching_bases = int(fields[9])
            alignment_length = int(fields[10])
            mapq = int(fields[11])

            if alignment_length == 0:
                continue

            identity = matching_bases / alignment_length

            # Score based on alignment identity and mapping quality
            score = identity * mapq

            if read_id not in minimap_best_results:
                minimap_best_results[read_id] = {
                    "predicted": target_name,
                    "identity": identity,
                    "mapq": mapq,
                    "score": score
                }

            elif score > minimap_best_results[read_id]["score"]:
                minimap_best_results[read_id] = {
                    "predicted": target_name,
                    "identity": identity,
                    "mapq": mapq,
                    "score": score
                }


# MINIMAP2 REPORT

minimap_counts = defaultdict(int)
minimap_correct = 0
minimap_total = 0

for read_id, result in minimap_best_results.items():
    actual = read_id.split("|")[0]
    predicted = result["predicted"]

    minimap_total += 1
    minimap_counts[predicted] += 1

    if predicted == actual:
        minimap_correct += 1

minimap_accuracy = (minimap_correct / minimap_total) * 100 if minimap_total > 0 else 0.0

print("\nMINIMAP2 CLASSIFICATION REPORT")
print(f"Total aligned reads: {minimap_total}")
print(f"Correctly classified: {minimap_correct}")
print(f"Accuracy: {minimap_accuracy:.2f}%")

print("\nMINIMAP2 METAGENOMIC SAMPLE COMPOSITION")
for genome, count in sorted(minimap_counts.items()):
    print(f"{genome}: {count} reads")


# COMPARISON BETWEEN K-MER AND MINIMAP2

kmer_dict = {}

for result in kmer_results:
    kmer_dict[result["read_id"]] = result["predicted"]

common_reads = 0
same_predictions = 0

for read_id, minimap_result in minimap_best_results.items():
    if read_id in kmer_dict:
        common_reads += 1

        if kmer_dict[read_id] == minimap_result["predicted"]:
            same_predictions += 1

agreement = (same_predictions / common_reads) * 100 if common_reads > 0 else 0.0

print("\nK-MER VS MINIMAP2 COMPARISON")
print(f"Common reads compared: {common_reads}")
print(f"Same predictions: {same_predictions}")
print(f"Agreement: {agreement:.2f}%")

minimap_end_time = time.time()

minimap_runtime = minimap_end_time - minimap_start_time

print(f"\nMINIMAP2 Runtime: {minimap_runtime:.2f} seconds")

Running minimap2 for each reference genome...
Running minimap2 for enterobacter_aerogenes...
Running minimap2 for escherichia_coli...
Running minimap2 for streptococcus_infantis...
Running minimap2 for salmonella_enterica...
Running minimap2 for blautia_coccoides...

MINIMAP2 CLASSIFICATION REPORT
Total aligned reads: 15360
Correctly classified: 15031
Accuracy: 97.86%

MINIMAP2 METAGENOMIC SAMPLE COMPOSITION
blautia_coccoides: 1996 reads
enterobacter_aerogenes: 189 reads
escherichia_coli: 4310 reads
salmonella_enterica: 4136 reads
streptococcus_infantis: 4729 reads

K-MER VS MINIMAP2 COMPARISON
Common reads compared: 15360
Same predictions: 14628
Agreement: 95.23%

MINIMAP2 Runtime: 24.58 seconds
